**🧱 What is the OPTIMIZE Command?**

- •	The OPTIMIZE command is used to compact small files in a Delta table into larger files. This helps improve query performance and reduce overhead.
- •	Think of it like cleaning up a folder that has hundreds of tiny files and combining them into a few big ones—much easier and faster to read!

**🧠 Why is File Compaction Important?**

When data is written in small batches (like streaming or frequent inserts), it creates many small files. These small files:
- 1.	Slow down queries
- 2.	Increase metadata load
- 3.	Make file scanning inefficient
- 4.	Compaction solves this by merging them into fewer, larger files.

________________________________________
**✅ Simple Example**
This command will compact the files in the sales_data Delta table.
OPTIMIZE sales_data;

🔍 How It Works in PySpark
You can also run it using SQL in a PySpark notebook:
spark.sql("OPTIMIZE sales_data")
This will:
- 1.	Scan the table
- 2.	Identify small files
- 3.	Merge them into larger files (default target size is ~1GB)
________________________________________
📦 Summary
Feature	Description
Purpose	Merge small files into larger ones
Benefit	Faster queries, less metadata overhead
Usage	OPTIMIZE table_name
Optional	ZORDER BY for clustering data
When to Use	After frequent inse


### create table

In [0]:
from delta.tables import *

DeltaTable.createOrReplace(spark) \
    .tableName("tri.tri.scd2Demo3") \
    .addColumn("pk1", "INT") \
    .addColumn("pk2", "STRING") \
    .addColumn("dim1", "INT") \
    .addColumn("dim2", "INT") \
    .addColumn("dim3", "INT") \
    .addColumn("dim4", "INT") \
    .addColumn("active_status", "STRING") \
    .addColumn("start_date", "TIMESTAMP") \
    .addColumn("end_date", "TIMESTAMP") \
    .execute()

In [0]:
%sql
insert into tri.tri.scd2demo3 values (111,'Unit1',200,500,800,400,'Y',current_timestamp(),'9999-12-31');

insert into tri.tri.scd2Demo3 values (222,'Unit2',900,NULL,700,100,'Y',current_timestamp(),'9999-12-31');

insert into tri.tri.scd2Demo3 values (333,'Unit3',300,900,250,650,'Y',current_timestamp(),'9999-12-31');

In [0]:
%sql
insert into tri.tri.scd2Demo3 values (666,'Unit6',200,500,800,400,'Y',current_timestamp(),'9999-12-31');

insert into tri.tri.scd2Demo3 values (777,'Unit7',900,NULL,700,100,'Y',current_timestamp(),'9999-12-31');

insert into tri.tri.scd2Demo3 values (888,'Unit8',300,900,250,650,'Y',current_timestamp(),'9999-12-31');

### Here we store the table into a particular location so we can perform operations on top of the path or location

In [0]:
spark.table("tri.tri.scd2Demo3") \
    .write \
    .format("delta") \
    .mode("append") \
    .save("/Volumes/tri/tri/tri_files/scd2Demo3")

In [0]:
%sql
SELECT * FROM delta.`/Volumes/tri/tri/tri_files/scd2Demo3`


### creating table instance with path

In [0]:
target_table=DeltaTable.forPath(spark, "/Volumes/tri/tri/tri_files/scd2Demo3")

In [0]:
display(target_table.toDF())


### timetravel with path+version


In [0]:
display(target_table.history())

### sql code

In [0]:
%sql
SELECT *
FROM delta.`/Volumes/tri/tri/tri_files/scd2Demo3`
VERSION AS OF 2;

### pyspark code
### 

In [0]:
df = spark.read \
    .format("delta") \
    .option("versionAsOf", 1) \
    .load("/Volumes/tri/tri/tri_files/scd2Demo3")

display(df)

total 17 rowss there -doing some dml operations

In [0]:
display(target_table.toDF())

### deleting using table instance


In [0]:
target_table.delete(
    condition = "pk1 = 777"
)

In [0]:
display(target_table.toDF())

### updating table using table instance

In [0]:
target_table.update(
    condition = "pk1 = 666",
    set = {
        "dim1": "100632222"
    }
)

In [0]:
display(target_table.toDF())

# doing optimize command

In [0]:
target_table.optimize().executeCompaction()

In [0]:
display(
    target_table.optimize().executeCompaction()
)

In [0]:
display(target_table.history())

### 1. List All Delta Table Data Files and Metadata Files

In [0]:
display(
    dbutils.fs.ls("/Volumes/tri/tri/tri_files/scd2Demo3")
)

### 2. List Delta Transaction Log Files (_delta_log)

In [0]:
display(
    dbutils.fs.ls("/Volumes/tri/tri/tri_files/scd2Demo3/_delta_log")
)

### 3. Read a Delta Log JSON File as Raw Text


In [0]:
display(
    spark.read.text(
        "dbfs:/Volumes/tri/tri/tri_files/scd2Demo3/_delta_log/00000000000000000039.json"
    )
)


### 4. Parse and Display Delta Log JSON File as Structured Data

In [0]:
display(
    spark.read.json(
        "dbfs:/Volumes/tri/tri/tri_files/scd2Demo3/_delta_log/00000000000000000039.json" ,multiLine=True
    )
)